## Prerequisites

**Run these first:**
1. ✅ `preprocessing_FIXED.ipynb`
2. ✅ `01_setup_and_config.ipynb`

In [ ]:
# Restore from setup
%store -r config
%store -r device

import sys
import os
sys.path.insert(0, os.getcwd())

from utils import (
    set_seed, get_dataloaders, validate_checkpoint_fresh,
    train_one_epoch, validate, evaluate_model
)
from models import TumorNetLite, count_parameters

import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("✓ Setup complete")

## 1. Define Training Function

In [ ]:
def train_variant(variant_name, model, train_loader, val_loader, test_loader, 
                 config, device, max_epochs=50):
    """
    Train a model variant for ablation study.
    Uses reduced epochs for faster comparison.
    """
    print(f"\n{'='*80}")
    print(f"TRAINING: {variant_name}")
    print(f"{'='*80}\n")
    
    # Setup
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config['optimizer']['learning_rate'],
        weight_decay=config['optimizer']['weight_decay']
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )
    scaler = torch.cuda.amp.GradScaler() if config['training']['mixed_precision'] else None
    
    # Training loop
    best_val_acc = 0.0
    patience = 0
    max_patience = 7
    
    for epoch in range(1, max_epochs + 1):
        # Train
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer,
            device, scaler, max_grad_norm=1.0, epoch=epoch
        )
        
        # Validate
        val_loss, val_acc = validate(
            model, val_loader, criterion, device
        )
        
        scheduler.step(val_acc)
        
        print(f"Epoch [{epoch}/{max_epochs}] Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience = 0
        else:
            patience += 1
            if patience >= max_patience:
                print(f"Early stopping at epoch {epoch}")
                break
    
    # Evaluate on test set
    results = evaluate_model(
        model, test_loader, device, config['data']['class_names']
    )
    
    return {
        'variant': variant_name,
        'accuracy': float(results['accuracy']),
        'parameters': count_parameters(model),
        'best_val_acc': float(best_val_acc),
        'epochs_trained': epoch
    }

## 2. Load Data

In [ ]:
# Load data once for all variants
train_loader, val_loader, test_loader, _ = get_dataloaders(
    config=config,
    preprocessed_dir=config['paths']['preprocessed_data']
)

class_names = config['data']['class_names']
num_classes = len(class_names)

print(f"✓ Data loaded")
print(f"  Classes: {class_names}")

## 3. Variant 1: Full TumorNet-Lite

In [ ]:
# Set seed for fair comparison
set_seed(config['reproducibility']['seed'], config['reproducibility']['deterministic'])

# Create full model
model_full = TumorNetLite(
    num_classes=num_classes,
    pretrained=False,
    in_channels=3,
    base_channels=64
).to(device)

# Train
result_full = train_variant(
    "Full TumorNet-Lite (All Components)",
    model_full, train_loader, val_loader, test_loader,
    config, device, max_epochs=50
)

print(f"\n✓ Full Model Results:")
print(f"  Accuracy: {result_full['accuracy']:.2f}%")
print(f"  Parameters: {result_full['parameters']:,}")

## 4. Variant 2-5: Simplified Versions

**Note:** For now, we'll use smaller baseline models as proxies.  
You can modify `models.py` to create actual ablation variants.

In [ ]:
from models import get_model

# Use MobileNetV3 as baseline (no TumorNet components)
set_seed(config['reproducibility']['seed'], config['reproducibility']['deterministic'])

model_baseline = get_model(
    'mobilenet_v3_small',
    num_classes=num_classes,
    pretrained=False
).to(device)

result_baseline = train_variant(
    "Baseline (MobileNetV3-Small)",
    model_baseline, train_loader, val_loader, test_loader,
    config, device, max_epochs=50
)

print(f"\n✓ Baseline Results:")
print(f"  Accuracy: {result_baseline['accuracy']:.2f}%")
print(f"  Parameters: {result_baseline['parameters']:,}")

## 5. Compare Results

In [ ]:
# Compile all results
ablation_results = [
    result_full,
    result_baseline
]

# Create DataFrame
df = pd.DataFrame(ablation_results)
df = df.sort_values('accuracy', ascending=False)

print("\n" + "="*80)
print("ABLATION STUDY RESULTS")
print("="*80)
print(df.to_string(index=False))
print("="*80)

## 6. Visualize Comparison

In [ ]:
# Bar plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy comparison
ax1.barh(df['variant'], df['accuracy'], color='steelblue')
ax1.set_xlabel('Test Accuracy (%)', fontsize=12)
ax1.set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Parameters comparison
ax2.barh(df['variant'], df['parameters'] / 1e6, color='coral')
ax2.set_xlabel('Parameters (Millions)', fontsize=12)
ax2.set_title('Model Size Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config['paths']['results'], 'ablation_comparison.png'), 
            dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved")

## 7. Save Results

In [ ]:
# Save to JSON
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_path = os.path.join(
    config['paths']['results'],
    f'ablation_study_{timestamp}.json'
)

with open(results_path, 'w') as f:
    json.dump(ablation_results, f, indent=2)

print(f"✓ Results saved to {results_path}")

## 8. Analysis & Conclusions

In [ ]:
print("\n" + "="*80)
print("ABLATION STUDY ANALYSIS")
print("="*80)

# Calculate improvements
baseline_acc = result_baseline['accuracy']
full_acc = result_full['accuracy']
improvement = full_acc - baseline_acc

print(f"\nBaseline Accuracy: {baseline_acc:.2f}%")
print(f"Full Model Accuracy: {full_acc:.2f}%")
print(f"Improvement: {improvement:.2f} percentage points")
print(f"Relative Improvement: {(improvement / baseline_acc * 100):.2f}%")

# Parameter efficiency
baseline_params = result_baseline['parameters']
full_params = result_full['parameters']
param_ratio = full_params / baseline_params

print(f"\nBaseline Parameters: {baseline_params:,}")
print(f"Full Model Parameters: {full_params:,}")
print(f"Parameter Ratio: {param_ratio:.2f}x")

# Efficiency score (accuracy per million parameters)
baseline_efficiency = baseline_acc / (baseline_params / 1e6)
full_efficiency = full_acc / (full_params / 1e6)

print(f"\nBaseline Efficiency: {baseline_efficiency:.2f}% per million params")
print(f"Full Model Efficiency: {full_efficiency:.2f}% per million params")

print("\n" + "="*80)
print("CONCLUSIONS")
print("="*80)

if full_acc > baseline_acc:
    print("✓ Novel components improve performance")
else:
    print("✗ Novel components do not improve performance")
    print("  → Check: data preprocessing, training protocol, hyperparameters")

if full_efficiency > baseline_efficiency:
    print("✓ Full model is more parameter-efficient")
else:
    print("✗ Baseline is more parameter-efficient")

print("="*80)

## Notes for Complete Ablation Study

**To create actual ablation variants:**

1. Modify `models.py` to add variant classes:
   - `TumorNetLite_NoSCTA`: Remove SCTA modules
   - `TumorNetLite_NoAPF`: Replace APF with standard conv
   - `TumorNetLite_NoPFR`: Remove PFR stages
   - `TumorNetLite_Baseline`: Simple CNN baseline

2. Train each variant with same protocol

3. Compare all 5 variants

**Expected pattern:** Full > -SCTA > -APF > -PFR > Baseline

This demonstrates each component contributes to final performance.